# 01 — Data handoff & fixed splits

**Phạm vi:** bước 1 nhận dữ liệu và bước 2 chia dữ liệu. Chưa train model, impute, encode hoặc scale.

Dataset: Vietnam Housing Dataset 2024; `Price` là tổng giá, **tỷ VND**. Chưa xác minh giá rao hay giá giao dịch. Mục tiêu: căn nhà khác trong bối cảnh tương tự dataset, không tuyên bố dự báo thị trường tương lai.

## Quy ước cell dùng từ nay
- `Mxx` = Markdown (câu hỏi, quyết định, kết luận); `Cxx` = Code (thực hiện một công việc).
- Dùng mã trong tiêu đề/comment, **không dùng số execution `[1]`, `[2]`** để gọi cell. Không đổi mã khi chạy lại hoặc chèn cell; nếu cần chèn dùng `C12a`.
- Hướng dẫn sau này sẽ ghi cả notebook và mã, ví dụ: “Trong notebook 01, sửa C02”.
- Chạy từ trên xuống; muốn kiểm tra độc lập thì Restart Kernel rồi Run All. Ghi kết luận ở Markdown, không trộn vào code.

| Mã | Công việc |
| --- | --- |
| C01 | Imports |
| C02 | Đường dẫn, config và đọc clean data |
| C11 | Schema + missing + 5 dòng mẫu |
| C12 | Target và thống kê tiếp nhận |
| C13 | Trùng thuộc tính sau rút gọn địa chỉ |
| M14 | Kết luận bước 1 |
| M20 | Giao thức split và metric |
| C21 | Tạo row_id để truy vết |
| C22 | Chia 4 tập |
| C23 | Bảng số dòng / tỷ lệ |
| C24 | Kiểm tra không giao row_id, không mất dòng |
| C25 | Lưu lần đầu hoặc xác minh bản đã lưu; không ghi đè |
| M26 | Điều kiện hoàn thành bước 2 |
| M30 | Lộ trình tiếp theo và mã cell sẽ dùng |


In [ ]:
# C01 — Imports
from pathlib import Path
from itertools import combinations
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split


In [ ]:
# C02 — Cấu hình và đọc đúng phiên bản dữ liệu
# Tìm root từ vị trí kernel (root repo hoặc thư mục notebooks).
PROJECT_ROOT = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "data" / "_v1.csv").is_file()
     and (folder / "notebooks").is_dir()),
    None,
)
assert PROJECT_ROOT is not None, "Mở kernel trong repo hoặc notebooks."

DATA_PATH = PROJECT_ROOT / "data" / "_v1.csv"
SPLIT_DIR = PROJECT_ROOT / "data" / "processed" / "splits_v1"
TARGET = "Price"
SEED = 42

NUMERIC_FEATURES = ["Area", "Frontage", "Access Road", "Floors", "Bedrooms", "Bathrooms"]
CATEGORICAL_FEATURES = [
    "House direction", "Balcony direction", "Legal status",
    "Furniture state", "City", "District",
]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

df = pd.read_csv(DATA_PATH)
assert df.shape == (30229, 13), "Data version khác handover; đối chiếu DE trước khi tiếp tục."
assert set(df.columns) == set(FEATURES + [TARGET]), "Schema khác handover."
print("Data:", DATA_PATH)
print("Shape:", df.shape)


## M10 — Bước 1: xác nhận dữ liệu đầu vào

**Câu hỏi:** target, schema, missing và đơn vị quan sát có đủ rõ để modeling không?

**Thao tác:** C11–C13 chỉ đọc và tổng hợp dữ liệu; không làm lại cleaning của DE. Output phục vụ kiểm tra handover, không phải train-only EDA để lựa chọn model. Phân tích quan hệ feature–target sâu hơn sẽ dùng train ở notebook baseline.


In [60]:
# C11 — Schema và missing: gom vào một bảng duy nhất
schema_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1),
})
display(schema_summary)
display(df.head(5))


,dtype,missing_count,missing_pct
Area,float64,0,0.0
Frontage,float64,0,0.0
Access Road,float64,0,0.0
House direction,str,21239,70.3
Balcony direction,str,24983,82.6
Floors,float64,3603,11.9
Bedrooms,float64,5162,17.1
Bathrooms,float64,7074,23.4
Legal status,str,4506,14.9
Furniture state,str,14119,46.7


,Area,Frontage,Access Road,House direction,Balcony direction,Floors,Bedrooms,Bathrooms,Legal status,Furniture state,Price,City,District
0,84.0,5.0,13.0,NaN,NaN,4.0,NaN,NaN,Have certificate,NaN,8.60,Hưng Yên,Văn Giang
1,60.0,5.0,13.0,NaN,NaN,5.0,NaN,NaN,NaN,NaN,7.50,Hưng Yên,Văn Giang
2,90.0,6.0,13.0,Đông - Bắc,Đông - Bắc,5.0,NaN,NaN,Sale contract,NaN,8.90,Hưng Yên,Văn Giang
3,54.0,4.1,3.5,Tây - Nam,Tây - Nam,2.0,2.0,3.0,Have certificate,Full,5.35,Hồ Chí Minh,Gò Vấp
4,92.0,4.1,5.0,Đông - Nam,Đông - Nam,2.0,4.0,4.0,Have certificate,Full,6.90,Hồ Chí Minh,Gò Vấp


In [61]:
# C12 — Target: kiểm tra và thống kê đúng một lần
assert pd.api.types.is_numeric_dtype(df[TARGET]), "Target phải là numeric."
assert df[TARGET].notna().all(), "Target có missing; trao đổi với DE."
assert df[TARGET].gt(0).all(), "Target có giá không dương; kiểm tra nguồn."
assert not df[TARGET].isin([float("inf"), -float("inf")]).any(), "Target có giá vô hạn."

print("Target:", TARGET, "| Đơn vị: tỷ VND")
display(df[TARGET].describe().to_frame("Price — tỷ VND"))


Target: Price | Đơn vị: tỷ VND


,Price — tỷ VND
count,30229.000000
mean,5.872078
std,2.211877
min,1.000000
25%,4.200000
50%,5.900000
75%,7.500000
max,11.500000


In [62]:
# C13 — Trùng toàn bộ cột sau rút gọn địa chỉ; KHÔNG drop_duplicates
duplicate_count = int(df.duplicated().sum())
print(f"Dòng lặp thêm trên 13 cột: {duplicate_count} ({duplicate_count / len(df):.2%})")
print("Đây là trùng thuộc tính, không tự chứng minh trùng căn nhà.")
# Không in lại toàn bộ các ví dụ địa chỉ; xem thêm chỉ khi cần điều tra.


Dòng lặp thêm trên 13 cột: 202 (0.67%)
Đây là trùng thuộc tính, không tự chứng minh trùng căn nhà.


## M14 — Kết luận bước 1

- Bản đã bàn giao: 30.229 dòng, 13 cột; target `Price`, tỷ VND, không thiếu nhãn.
- Theo xác nhận của team: mỗi dòng là một căn nhà riêng; raw không trùng toàn dòng. Sau rút gọn địa chỉ còn City/District có 202 dòng lặp thêm trên 13 cột. **Giữ nguyên các dòng này**. Phép kiểm tra duplicate riêng lẻ không chứng minh định danh căn nhà.
- 6 numeric và 6 categorical là feature ứng viên. Missing của hướng nhà/ban công cao; không tự drop dòng hay cột chỉ vì tỷ lệ này.
- DE xác nhận chưa median-impute/encode, không lọc giá thêm hoặc deduplicate trước handover. Không suy ra tác giả Kaggle chưa xử lý dữ liệu; trạng thái scaling/biến đổi khác cần đối chiếu nếu phát sinh nghi vấn.
- Giá rao hay giá giao dịch và quy trình upstream chưa rõ. Không gọi output là giá giao dịch thực tế hoặc khẳng định dữ liệu hoàn toàn không leakage.
- Nếu C11–C13 khớp ghi nhận trên: hoàn tất rà soát bước 1. Nếu khác, ghi vấn đề và đối chiếu bản dữ liệu; không sửa dữ liệu chỉ để khớp con số.


## M20 — Bước 2: cố định giao thức đánh giá

**Vì sao random split:** mỗi dòng được team xác nhận là căn nhà riêng, mục tiêu là căn khác trong bối cảnh tương tự; không đánh giá tương lai hoặc khu vực hoàn toàn mới.

| Tập | Tỷ lệ mục tiêu | Vai trò |
| --- | --- | --- |
| Train | 60% | Fit preprocessing và model |
| Validation | 15% | Chọn feature, model, hyperparameter |
| Calibration | 10% | Hiệu chỉnh interval sau khi khóa model |
| Test | 15% | Đánh giá cuối một lần theo kế hoạch |

- Seed = 42, không đổi seed để tìm điểm đẹp. Tỷ lệ thực tế làm tròn theo số dòng.
- MAE validation là metric chính; RMSE theo dõi lỗi lớn, R² theo dõi yêu cầu. Tiêu chí nghiệm thu: **test R² ≥ 0,7**, không phải train.
- MAE/RMSE dùng tỷ VND; chưa tính metrics ở notebook này.
- `row_id` chỉ truy vết dòng của data version hiện tại, không đưa vào X và không chứng minh danh tính căn nhà.
- Chưa học median, encoder, scaler trước split. Không phân tích calibration/test để chọn model.


In [63]:
# C21 — Tạo bản split có row_id; giữ df nguyên vẹn
data = df.copy().reset_index(drop=True)
assert "row_id" not in data.columns
data.insert(0, "row_id", range(len(data)))
assert data["row_id"].is_unique
print(f"Sẵn sàng chia {len(data):,} dòng; row_id không phải feature.")


Sẵn sàng chia 30,229 dòng; row_id không phải feature.


In [64]:
# C22 — Giữ nguyên chiến lược và seed đã dùng trong notebook cũ
remaining_df, test_df = train_test_split(
    data, test_size=0.15, random_state=SEED, shuffle=True,
)
development_df, calibration_df = train_test_split(
    remaining_df, test_size=10 / 85, random_state=SEED, shuffle=True,
)
train_df, val_df = train_test_split(
    development_df, test_size=15 / 75, random_state=SEED, shuffle=True,
)

splits = {
    "train": train_df,
    "validation": val_df,
    "calibration": calibration_df,
    "test": test_df,
}


In [65]:
# C23 — Chỉ kiểm tra kích thước, không EDA giá trên test/calibration
split_summary = pd.DataFrame([
    {"split": name, "rows": len(part),
     "percentage": round(len(part) / len(data) * 100, 2)}
    for name, part in splits.items()
])
display(split_summary)
print("Tổng số dòng:", split_summary["rows"].sum())


,split,rows,percentage
0,train,18136,60.0
1,validation,4535,15.0
2,calibration,3023,10.0
3,test,4535,15.0


Tổng số dòng: 30229


In [66]:
# C24 — Kiểm tra phân hoạch theo row_id
for name, part in splits.items():
    assert part["row_id"].is_unique, f"{name}: row_id bị lặp."

for name_a, name_b in combinations(splits, 2):
    assert set(splits[name_a]["row_id"]).isdisjoint(splits[name_b]["row_id"]), (
        f"{name_a} và {name_b} giao row_id."
    )

all_ids = set().union(*(set(part["row_id"]) for part in splits.values()))
assert all_ids == set(data["row_id"]), "Thiếu hoặc có row_id ngoài nguồn."
assert sum(map(len, splits.values())) == len(data), "Tổng số dòng không khớp."
print("PASS: không mất dòng, không giao row_id giữa 4 tập.")


PASS: không mất dòng, không giao row_id giữa 4 tập.


In [67]:
# C25 — Lưu lần đầu; chạy lại chỉ xác minh, KHÔNG ghi đè split cũ
manifest = pd.concat(
    [part[["row_id"]].assign(split=name) for name, part in splits.items()],
    ignore_index=True,
).sort_values("row_id").reset_index(drop=True)
tables_to_save = {**splits, "split_manifest": manifest}

if SPLIT_DIR.exists():
    for name, expected in tables_to_save.items():
        path = SPLIT_DIR / f"{name}.csv"
        assert path.is_file(), f"Thiếu {path.name}; chưa xác nhận split, không ghi đè."
        saved = pd.read_csv(path)
        # Cho phép khác dtype do CSV và sai số rất nhỏ khi đọc float.
        pd.testing.assert_frame_equal(
            saved.reset_index(drop=True), expected.reset_index(drop=True),
            check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12,
        )
    print("PASS: 5 file đã lưu khớp dữ liệu và split hiện tại; không ghi đè.")
else:
    SPLIT_DIR.mkdir(parents=True)
    for name, table in tables_to_save.items():
        table.to_csv(SPLIT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
    print("Đã lưu mới 4 tập và split_manifest.csv.")

print("Split directory:", SPLIT_DIR)


PASS: 5 file đã lưu khớp dữ liệu và split hiện tại; không ghi đè.
Split directory: d:\MinhTriet\LT\AIQS\AIQS-Real-Estate-Price-Prediction\data\processed\splits_v1


## M26 — Kết luận bước 2 / điểm dừng notebook

**Chỉ xác nhận bước 2 hoàn tất khi:** C24 PASS và C25 lưu thành công hoặc xác minh đủ 5 file đã có.

Kích thước dự kiến với version này: train 18.136; validation 4.535; calibration 3.023; test 4.535. Nếu output khác, điều tra data version/config; không đổi seed để ép khớp.

**Diễn giải:** các tập không giao dòng và tái lập được từ cùng đầu vào/thứ tự dòng. Điều này không tự bảo đảm không có mọi loại leakage. Nếu dữ liệu thay đổi, tạo version mới có ghi nhận, không ghi đè splits_v1.

**Bước tiếp theo:** mở notebook baseline; chỉ đọc train.csv và validation.csv đã lưu. Không cần chia lại ở mỗi experiment.


## M30 — Kế hoạch tiếp theo: làm ở đâu, sửa cell nào?

Notebook `02_baseline_models.ipynb` đã được tách từ các cell baseline bạn thực hiện trong file này. Các notebook experiments/final evaluation bên dưới mới là kế hoạch. evaluatePrize.ipynb hiện có không thuộc phạm vi chỉnh sửa lần này.

### Bước 3 — 02_baseline_models.ipynb
| Mã cell | Công việc / output |
| --- | --- |
| M00 | Mục tiêu, quy ước cell và metric |
| C01 | Imports |
| C02 | Đọc train/validation từ splits_v1; không đọc test/calibration |
| C03 | Feature allowlist, tách X/y, loại row_id và Price khỏi X |
| C04 | Hàm MAE/RMSE/R² và bảng log kết quả |
| C05 | Dummy median: mốc sai số ban đầu |
| C06 | Numeric/categorical preprocessing + Linear Regression pipeline |
| C07 | Fit trên train, predict và ghi metrics train/validation |
| M08 | Bạn đọc kết quả: có hơn Dummy không? Cần kiểm tra gì? |
| C09 | RF đối chứng: cùng feature/split, preprocessing riêng |
| C10 | Bảng comparison và lỗi validation cơ bản |
| M11 | Nhận xét, giới hạn và quyết định thử gì tiếp |

Không chạy RF/tuning chỉ để có thêm model trước khi hiểu output baseline. Median/encoder/scaler chỉ fit trên train.

### Bước 4 — 03_model_experiments.ipynb (khi đã đọc baseline)
Khối đầu C01–C03: đọc split cố định và thiết lập dùng chung. Mỗi experiment dùng 3 cell: `M10` hypothesis + yếu tố giữ nguyên, `C11` thực hiện/đánh giá, `M12` kết luận + next decision; experiment tiếp dùng M20/C21/M22. FE hoặc tuning cần một câu hỏi, không mặc định phải chạy mọi phương án. Project yêu cầu FE nên có một thử nghiệm có lý do; không bắt buộc giữ feature không cải thiện.

### Bước 5 — 04_final_evaluation.ipynb (sau khi khóa cấu hình)
M00: model selection; C01–C02: imports/config; C03: refit train + validation; C04: calibration cho interval; C05: final test MAE/RMSE/R², coverage và width; C06: export toàn bộ pipeline/calibration/metadata; C07: kiểm tra save/load và inference; M08: kết luận đạt/chưa đạt và hạn chế. Không tuning theo kết quả final test.

**Cách hỏi mentor:** gửi tên notebook + mã cell + output + cách bạn hiểu kết quả. Ví dụ: “02_baseline_models, C07: đây là bảng metrics; mình thấy ...”.
